# 03 — Reversal Signals

Analysis of short-term mean-reversion signals. Compares raw, volume-filtered, and activity-gated variants.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Auto-generate synthetic data if none exists
from statarb.utils import load_config
from statarb.data.storage import DataStore
from experiments.generate_synthetic_data import generate_all

cfg = load_config("../experiments/config.yaml")
data_cfg = cfg.get("data", {})
syn_cfg = cfg.get("synthetic", {})
exchange = data_cfg.get("exchange", "binance")
base_dir = data_cfg.get("base_dir", "../data")

store = DataStore(base_dir=base_dir)
symbols = store.list_symbols(exchange, "1d")
if not symbols:
    print("Generating synthetic data...")
    generate_all(
        n_assets=syn_cfg.get("n_assets", 20),
        n_days=syn_cfg.get("n_days", 1000),
        start_date=syn_cfg.get("start_date", "2021-01-01"),
        seed=syn_cfg.get("seed", 42),
        exchange=exchange,
        base_dir=base_dir,
    )
    symbols = store.list_symbols(exchange, "1d")

prices  = store.build_panel(symbols, exchange, "1d", field="close")
volume  = store.build_volume_panel(symbols, exchange, "1d")

from statarb.data.features import FeatureEngine
fe = FeatureEngine()
returns = fe.log_returns(prices)
vol_ratio = fe.volume_ma_ratio(volume, window=21)

print(f"Loaded: {len(symbols)} assets x {len(prices)} days")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")


## Signal Construction

In [ ]:
from statarb.signals.reversal import ReversalSignals
from statarb.signals.activity import ActivityFilter
rev = ReversalSignals()
act = ActivityFilter()

signals = {
    "reversal_1d":        rev.short_term_reversal(returns, lookback=1),
    "reversal_3d":        rev.short_term_reversal(returns, lookback=3),
    "reversal_5d":        rev.weekly_reversal(returns),
    "reversal_7d":        rev.short_term_reversal(returns, lookback=7),
    "vol_adj_reversal":   rev.vol_adjusted_reversal(returns),
    "bollinger_20d":      rev.bollinger_reversal(prices, window=20),
    "large_move_rev":     rev.large_move_reversal(returns),
    "vol_filtered_rev":       rev.volume_filtered_reversal(returns, volume, lookback=3),
}
ranked = {name: fe.cross_sectional_rank(sig) for name, sig in signals.items()}
gated = {
    name: fe.cross_sectional_rank(act.activity_gate(sig, volume, min_ratio=0.5))
    for name, sig in signals.items()
}
print("Signals built:", list(signals.keys()))


## Reversal Signal Decay

In [ ]:
from experiments._utils import signal_decay_ic

max_h = 15
decay = {name: signal_decay_ic(ranked[name], returns, max_h)
         for name in ["reversal_1d", "reversal_5d", "vol_adj_reversal", "large_move_rev"]}

fig, ax = plt.subplots(figsize=(10, 4))
for name, ic in decay.items():
    ax.plot(ic.index, ic.values, marker="o", markersize=4, linewidth=1.5, label=name)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Forward Horizon (days)")
ax.set_ylabel("Rank IC (Spearman)")
ax.set_title("Reversal Signal Decay", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Limit Orders vs Market Orders

In [ ]:
from experiments._utils import backtest_signals
from statarb.backtest.execution import ExecutionModel

em_market = ExecutionModel(market_order_cost=0.0020, order_type="market")
em_limit  = ExecutionModel(limit_order_cost=0.0007, order_type="limit")

res_market = backtest_signals(ranked, returns, em_market, 365)
res_limit  = backtest_signals(ranked, returns, em_limit,  365)

fig, ax = plt.subplots(figsize=(11, 4))
x = range(len(ranked))
ax.bar([i - 0.2 for i in x], res_market["sharpe"].values, width=0.35,
       label="Market (20bps)", alpha=0.7, color="tab:red")
ax.bar([i + 0.2 for i in x], res_limit["sharpe"].values, width=0.35,
       label="Limit (7bps)", alpha=0.7, color="tab:blue")
ax.set_xticks(list(x))
ax.set_xticklabels(list(ranked.keys()), rotation=45, ha="right", fontsize=8)
ax.set_title("Reversal Sharpe: Market vs Limit Orders", fontweight="bold")
ax.set_ylabel("Annualized Sharpe")
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print("Key insight: reversal signals need limit orders — market orders erase the alpha.")


## Volume Filter Analysis

In [ ]:
# Does high volume indicate more informed flow? If so, 1-day reversal
# should be weaker (momentum) on high-volume days.
fwd1 = returns.shift(-1)
rev1_ranked = ranked["reversal_1d"]
activity = vol_ratio.mean(axis=1)

# Split IC by activity quintile
quintiles = pd.qcut(activity.dropna(), 5, labels=["Q1 (low)", "Q2", "Q3", "Q4", "Q5 (high)"])
ic_by_quintile = {}
for q in quintiles.cat.categories:
    mask = (quintiles == q).reindex(rev1_ranked.index).fillna(False)
    dates = rev1_ranked.index[mask]
    ics = []
    for d in dates:
        if d not in fwd1.index:
            continue
        s = rev1_ranked.loc[d].dropna()
        r = fwd1.loc[d].reindex(s.index).dropna()
        s2 = s.reindex(r.index)
        if len(s2) >= 5:
            ics.append(float(s2.corr(r, method="spearman")))
    ic_by_quintile[q] = np.mean(ics) if ics else np.nan

print("1-Day Reversal IC by Volume Activity Quintile:")
for q, ic in ic_by_quintile.items():
    print(f"  {q}: {ic:.4f}")
print("Negative on high-volume days → momentum effect dominates (informed flow)")
